# Regression Prediction: eval_users1000

Summary notebook for MovieLens-1M rating-regression runs. It reads compact provenance from `outputs/in_distribution/regression_prediction/`, keeps only runs listed in `final_results.md`, and reports seed-level plus seed-averaged metrics. The parsing is intentionally local to this notebook because regression artifacts are still small and method-specific.

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", None)

def find_repo_root(start: Path | None = None) -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "pyproject.toml").exists() and candidate.name == "beyond-click-sim":
            return candidate
    raise RuntimeError("Could not find beyond-click-sim repo root")

REPO_ROOT = find_repo_root()
RESULTS_ROOT = REPO_ROOT / "outputs" / "in_distribution" / "regression_prediction"
FINAL_RESULTS_PATH = REPO_ROOT / "final_results.md"
RESULTS_ROOT

PosixPath('/Users/a.agafonov/Research/agent_simulator/repos/beyond-click-sim/outputs/in_distribution/regression_prediction')

In [2]:
TASK_RE = re.compile(
    r"^(?P<dataset>.+)_(?P<target>.+)_eval_users(?P<eval_users>\d+)_seed(?P<seed>\d+)$"
)
RUN_TS_RE = re.compile(r"^(?P<timestamp>\d{8}T\d{6}Z)_")
FINAL_RESULT_RUN_RE = re.compile(r"`(?P<run>\d{8}T\d{6}Z_[^`]+)`")

def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))

def as_dict(value: Any) -> dict[str, Any]:
    return value if isinstance(value, dict) else {}

def timestamp_from_run_dir(run_dir: Path) -> str:
    match = RUN_TS_RE.match(run_dir.name)
    return "" if match is None else match.group("timestamp")

def parse_task_name(task: str) -> dict[str, str | int]:
    match = TASK_RE.match(task)
    if match is None:
        return {"dataset": "unknown", "target": "unknown", "eval_users": -1, "seed": -1}
    data = match.groupdict()
    return {
        "dataset": data["dataset"],
        "target": data["target"],
        "eval_users": int(data["eval_users"]),
        "seed": int(data["seed"]),
    }

def metric_value(metrics: dict[str, Any], split: str, aggregation: str, metric: str) -> float | int | None:
    value = as_dict(as_dict(metrics.get(split)).get(aggregation)).get(metric)
    return value.item() if hasattr(value, "item") else value

def collect_final_result_runs() -> set[str]:
    text = FINAL_RESULTS_PATH.read_text(encoding="utf-8")
    return {match.group("run") for match in FINAL_RESULT_RUN_RE.finditer(text)}

def collect_regression_runs() -> pd.DataFrame:
    final_result_runs = collect_final_result_runs()
    rows: list[dict[str, Any]] = []
    for metrics_path in sorted(RESULTS_ROOT.glob("*/metrics.json")):
        run_dir = metrics_path.parent
        if run_dir.name not in final_result_runs:
            continue
        manifest_path = run_dir / "manifest.json"
        if not manifest_path.exists():
            continue
        metrics = read_json(metrics_path)
        manifest = read_json(manifest_path)
        if metrics.get("protocol") != "regression":
            continue

        task = str(metrics.get("task", ""))
        task_info = parse_task_name(task)
        task_manifest = as_dict(as_dict(manifest.get("task")).get("manifest"))
        scorer = as_dict(manifest.get("scorer"))
        history_window = as_dict(scorer.get("history_window"))
        llm_target = as_dict(scorer.get("target"))
        manifest_rows = as_dict(task_manifest.get("rows"))
        manifest_users = as_dict(task_manifest.get("users"))

        rows.append({
            "timestamp": timestamp_from_run_dir(run_dir),
            "run": run_dir.name,
            "task": task,
            "method": metrics.get("method"),
            "dataset": task_info["dataset"],
            "target": task_info["target"],
            "eval_users": task_info["eval_users"],
            "seed": task_info["seed"],
            "target_source_column": task_manifest.get("target_source_column"),
            "train_rows": manifest_rows.get("train"),
            "val_rows": manifest_rows.get("val"),
            "test_rows": manifest_rows.get("test"),
            "train_users": manifest_users.get("train"),
            "val_users": manifest_users.get("val"),
            "test_users": manifest_users.get("test"),
            "scorer_class": scorer.get("class"),
            "scorer_mean": scorer.get("mean"),
            "scorer_mode": scorer.get("mode"),
            "history_max_items": history_window.get("max_history_items"),
            "client_name": scorer.get("client_name"),
            "model": scorer.get("model"),
            "llm_target_name": llm_target.get("name"),
            "requested_rows": metrics.get("requested_rows"),
            "scored_rows": metrics.get("scored_rows"),
            "coverage": metrics.get("coverage"),
            "llm_errors": metrics.get("llm_errors"),
            "val_metric_rows": metric_value(metrics, "val", "micro", "n"),
            "val_metric_users": metric_value(metrics, "val", "macro_by_user_mean", "n_users"),
            "test_metric_rows": metric_value(metrics, "test", "micro", "n"),
            "test_metric_users": metric_value(metrics, "test", "macro_by_user_mean", "n_users"),
            "val_macro_mae": metric_value(metrics, "val", "macro_by_user_mean", "mae"),
            "val_macro_rmse": metric_value(metrics, "val", "macro_by_user_mean", "rmse"),
            "test_macro_mae": metric_value(metrics, "test", "macro_by_user_mean", "mae"),
            "test_macro_rmse": metric_value(metrics, "test", "macro_by_user_mean", "rmse"),
            "test_micro_mae": metric_value(metrics, "test", "micro", "mae"),
            "test_micro_rmse": metric_value(metrics, "test", "micro", "rmse"),
            "main_metric": metrics.get("main_metric"),
        })
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["dataset", "target", "method", "seed", "timestamp", "run"]).reset_index(drop=True)

all_runs = collect_regression_runs()
latest_runs = (
    all_runs.sort_values(["task", "method", "timestamp", "run"])
    .groupby(["task", "method"], as_index=False, dropna=False)
    .tail(1)
    .sort_values(["dataset", "target", "method", "seed"])
    .reset_index(drop=True)
)
latest_runs[["dataset", "target", "method", "seed", "timestamp", "run"]]

,dataset,target,method,seed,timestamp,run
0,ml-1m,rating,item_mean_regressor,0,20260618T084646Z,20260618T084646Z_ml-1m_rating_eval_users1000_seed0_item_mean_regressor
1,ml-1m,rating,item_mean_regressor,1,20260618T084659Z,20260618T084659Z_ml-1m_rating_eval_users1000_seed1_item_mean_regressor
2,ml-1m,rating,item_mean_regressor,2,20260618T084712Z,20260618T084712Z_ml-1m_rating_eval_users1000_seed2_item_mean_regressor
3,ml-1m,rating,item_mean_regressor,3,20260618T084725Z,20260618T084725Z_ml-1m_rating_eval_users1000_seed3_item_mean_regressor
4,ml-1m,rating,item_mean_regressor,4,20260618T084738Z,20260618T084738Z_ml-1m_rating_eval_users1000_seed4_item_mean_regressor
5,ml-1m,rating,item_mode_regressor,0,20260618T084647Z,20260618T084647Z_ml-1m_rating_eval_users1000_seed0_item_mode_regressor
6,ml-1m,rating,item_mode_regressor,1,20260618T084700Z,20260618T084700Z_ml-1m_rating_eval_users1000_seed1_item_mode_regressor
7,ml-1m,rating,item_mode_regressor,2,20260618T084713Z,20260618T084713Z_ml-1m_rating_eval_users1000_seed2_item_mode_regressor
8,ml-1m,rating,item_mode_regressor,3,20260618T084726Z,20260618T084726Z_ml-1m_rating_eval_users1000_seed3_item_mode_regressor
9,ml-1m,rating,item_mode_regressor,4,20260618T084739Z,20260618T084739Z_ml-1m_rating_eval_users1000_seed4_item_mode_regressor


## Latest Runs

In [3]:
latest_columns = [
    "dataset", "target", "method", "seed",
    "train_rows", "val_metric_rows", "test_metric_rows", "test_metric_users",
    "requested_rows", "scored_rows", "coverage", "llm_errors",
    "scorer_mean", "scorer_mode", "history_max_items",
    "test_macro_mae", "test_macro_rmse", "test_micro_mae", "test_micro_rmse",
    "run",
]
latest_table = latest_runs[latest_columns]

display(
    latest_table.style
    .format({
        "coverage": "{:.3f}",
        "scorer_mean": "{:.4f}",
        "scorer_mode": "{:.4f}",
        "test_macro_mae": "{:.4f}",
        "test_macro_rmse": "{:.4f}",
        "test_micro_mae": "{:.4f}",
        "test_micro_rmse": "{:.4f}",
    }, na_rep="")
    .hide(axis="index")
)

dataset,target,method,seed,train_rows,val_metric_rows,test_metric_rows,test_metric_users,requested_rows,scored_rows,coverage,llm_errors,scorer_mean,scorer_mode,history_max_items,test_macro_mae,test_macro_rmse,test_micro_mae,test_micro_rmse,run
ml-1m,rating,item_mean_regressor,0,694335,17629.000000,34520,1000,,,,,,,,0.7912,0.9560,0.7755,0.9710,20260618T084646Z_ml-1m_rating_eval_users1000_seed0_item_mean_regressor
ml-1m,rating,item_mean_regressor,1,694335,16574.000000,32425,1000,,,,,,,,0.7782,0.9397,0.7852,0.9826,20260618T084659Z_ml-1m_rating_eval_users1000_seed1_item_mean_regressor
ml-1m,rating,item_mean_regressor,2,694335,16731.000000,32738,1000,,,,,,,,0.7825,0.9429,0.7807,0.9783,20260618T084712Z_ml-1m_rating_eval_users1000_seed2_item_mean_regressor
ml-1m,rating,item_mean_regressor,3,694335,17388.000000,34008,1000,,,,,,,,0.7905,0.9543,0.7730,0.9692,20260618T084725Z_ml-1m_rating_eval_users1000_seed3_item_mean_regressor
ml-1m,rating,item_mean_regressor,4,694335,17564.000000,34407,1000,,,,,,,,0.7982,0.9642,0.7920,0.9925,20260618T084738Z_ml-1m_rating_eval_users1000_seed4_item_mean_regressor
ml-1m,rating,item_mode_regressor,0,694335,17629.000000,34520,1000,,,,,,,,0.7782,1.0619,0.7644,1.0850,20260618T084647Z_ml-1m_rating_eval_users1000_seed0_item_mode_regressor
ml-1m,rating,item_mode_regressor,1,694335,16574.000000,32425,1000,,,,,,,,0.7621,1.0427,0.7729,1.0962,20260618T084700Z_ml-1m_rating_eval_users1000_seed1_item_mode_regressor
ml-1m,rating,item_mode_regressor,2,694335,16731.000000,32738,1000,,,,,,,,0.7630,1.0432,0.7725,1.0953,20260618T084713Z_ml-1m_rating_eval_users1000_seed2_item_mode_regressor
ml-1m,rating,item_mode_regressor,3,694335,17388.000000,34008,1000,,,,,,,,0.7716,1.0592,0.7619,1.0862,20260618T084726Z_ml-1m_rating_eval_users1000_seed3_item_mode_regressor
ml-1m,rating,item_mode_regressor,4,694335,17564.000000,34407,1000,,,,,,,,0.7797,1.0626,0.7790,1.1050,20260618T084739Z_ml-1m_rating_eval_users1000_seed4_item_mode_regressor


## Seed-Averaged Summary

In [4]:
summary = (
    latest_runs.groupby(["dataset", "target", "eval_users", "method"], dropna=False)
    .agg(
        n_seeds=("seed", "nunique"),
        seeds=("seed", lambda values: tuple(sorted(int(value) for value in values.dropna()))),
        train_rows=("train_rows", "first"),
        test_metric_rows_mean=("test_metric_rows", "mean"),
        test_metric_users_mean=("test_metric_users", "mean"),
        requested_rows_mean=("requested_rows", "mean"),
        scored_rows_mean=("scored_rows", "mean"),
        coverage_mean=("coverage", "mean"),
        llm_errors_sum=("llm_errors", "sum"),
        test_macro_mae_mean=("test_macro_mae", "mean"),
        test_macro_mae_std=("test_macro_mae", "std"),
        test_macro_rmse_mean=("test_macro_rmse", "mean"),
        test_macro_rmse_std=("test_macro_rmse", "std"),
        test_micro_mae_mean=("test_micro_mae", "mean"),
        test_micro_mae_std=("test_micro_mae", "std"),
    )
    .reset_index()
)

display(
    summary.style
    .format({
        "test_metric_rows_mean": "{:.1f}",
        "test_metric_users_mean": "{:.1f}",
        "requested_rows_mean": "{:.1f}",
        "scored_rows_mean": "{:.1f}",
        "coverage_mean": "{:.3f}",
        "test_macro_mae_mean": "{:.4f}",
        "test_macro_mae_std": "{:.4f}",
        "test_macro_rmse_mean": "{:.4f}",
        "test_macro_rmse_std": "{:.4f}",
        "test_micro_mae_mean": "{:.4f}",
        "test_micro_mae_std": "{:.4f}",
    }, na_rep="")
    .hide(axis="index")
)

dataset,target,eval_users,method,n_seeds,seeds,train_rows,test_metric_rows_mean,test_metric_users_mean,requested_rows_mean,scored_rows_mean,coverage_mean,llm_errors_sum,test_macro_mae_mean,test_macro_mae_std,test_macro_rmse_mean,test_macro_rmse_std,test_micro_mae_mean,test_micro_mae_std
ml-1m,rating,1000,item_mean_regressor,5,"(0, 1, 2, 3, 4)",694335,33619.6,1000.0,,,,0.000000,0.7881,0.0079,0.9514,0.0100,0.7813,0.0076
ml-1m,rating,1000,item_mode_regressor,5,"(0, 1, 2, 3, 4)",694335,33619.6,1000.0,,,,0.000000,0.7709,0.0082,1.0539,0.0101,0.7701,0.0069
ml-1m,rating,1000,llm_regressor_vllm_llama33_70b_full,1,"(0,)",694335,34520.0,1000.0,34520.0,34520.0,1.000,0.000000,0.7995,,1.0804,,0.8057,
ml-1m,rating,1000,mean_regressor,5,"(0, 1, 2, 3, 4)",694335,33619.6,1000.0,,,,0.000000,0.9375,0.0033,1.0824,0.0036,0.9294,0.0071
ml-1m,rating,1000,mode_regressor,1,"(0,)",694335,34520.0,1000.0,,,,0.000000,0.8364,,1.0932,,0.8468,
ml-1m,rating,1000,user_mean_regressor,5,"(0, 1, 2, 3, 4)",694335,33619.6,1000.0,,,,0.000000,0.8435,0.0058,1.0221,0.0066,0.8451,0.0106
ml-1m,rating,1000,user_mode_regressor,5,"(0, 1, 2, 3, 4)",694335,33619.6,1000.0,,,,0.000000,0.8948,0.0105,1.1926,0.0098,0.8960,0.0147
ml-1m_rating_item,stats,1000,llm_regressor_vllm_llama33_70b_with_item_stats_full,1,"(0,)",694335,34520.0,1000.0,34520.0,34520.0,1.000,0.000000,0.7523,,1.0277,,0.7492,


## LLM Run Scope

In [5]:
llm_scope = latest_runs[latest_runs["method"].astype(str).str.startswith("llm_")][[
    "method", "seed", "test_rows", "test_metric_rows", "test_metric_users",
    "requested_rows", "scored_rows", "coverage", "llm_errors", "run",
]]
display(
    llm_scope.style
    .format({"coverage": "{:.3f}"}, na_rep="")
    .hide(axis="index")
)

method,seed,test_rows,test_metric_rows,test_metric_users,requested_rows,scored_rows,coverage,llm_errors,run
llm_regressor_vllm_llama33_70b_full,0,34520,34520,1000,34520.000000,34520.000000,1.000,0.000000,20260615T103530Z_ml-1m_rating_eval_users1000_seed0_llm_regressor_vllm_llama33_70b_full
llm_regressor_vllm_llama33_70b_with_item_stats_full,0,34520,34520,1000,34520.000000,34520.000000,1.000,0.000000,20260616T141222Z_ml-1m_rating_item_stats_eval_users1000_seed0_llm_regressor_vllm_llama33_70b_with_item_stats_full
